In [ ]:
# Check GPU
import torch
torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

In [ ]:
# Install deps (Colab)
%pip -q install --upgrade pip
%pip -q install torchvision pyyaml ultralytics

In [ ]:
# Create project structure and write the RCNN training script
import os, textwrap, pathlib
root = pathlib.Path('/content/ProjectEPICS')
script_dir = root / 'microplastic-ml' / 'scripts'
models_dir = root / 'microplastic-ml' / 'models'
runs_dir = root / 'runs' / 'rcnn'
script_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)
runs_dir.mkdir(parents=True, exist_ok=True)
script_path = script_dir / 'train_faster_rcnn.py'
script_content = r'''REPLACE_SCRIPT_CONTENT'''
script_path.write_text(script_content, encoding='utf-8')
print('Wrote', script_path)

## Bring your dataset
Provide a YOLOv8 dataset with the usual structure under a folder like `/content/microplastic_1153.v2.yolov8/`.
- It should contain `train/images`, `train/labels`, `valid/images`, `valid/labels`, and a `data.yaml`.
- Option A: Upload a ZIP to the Colab session and unzip to `/content`.
- Option B: Mount Google Drive and point `DATA_YAML` below to your Drive path.

In [ ]:
# Optional: mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Set your dataset's data.yaml path here
DATA_YAML = '/content/microplastic_1153.v2.yolov8/data.yaml'  # <-- change if different
import pathlib, json
p = pathlib.Path(DATA_YAML)
assert p.exists(), f'data.yaml not found at {p}. Please update DATA_YAML.'
print('Using data.yaml:', p)

In [ ]:
# Quick 1-epoch smoke test (adjust workers/batch as GPU allows)
import subprocess, sys
cmd = [sys.executable, str(script_path), '--data', DATA_YAML, '--epochs', '1', '--batch', '2', '--workers', '4', '--name', 'rcnn_quick', '--log-file', str(runs_dir / 'rcnn_quick' / 'train.log')]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('Done. Best weights should be at', models_dir / 'rcnn_best.pth')

In [ ]:
# Longer training (example 60 epochs) — uncomment to run
# import subprocess, sys
# cmd = [sys.executable, str(script_path), '--data', DATA_YAML, '--epochs', '60', '--batch', '2', '--workers', '4', '--name', 'rcnn_v1', '--log-file', str(runs_dir / 'rcnn_v1' / 'train.log')]
# print('Running:', ' '.join(cmd))
# subprocess.run(cmd, check=True)
# print('Done. Best weights at', models_dir / 'rcnn_best.pth')

In [ ]:
# COCO Faster R-CNN training on Colab
# 1) Install deps  2) Mount Drive  3) Write script  4) Run  5) Save to Drive
import sys, torch
print(torch.__version__)
!pip -q install pycocotools torchvision


In [ ]:
# Mount Google Drive (choose account when prompted)
from google.colab import drive
DRIVE_ROOT = "/content/drive"
drive.mount(DRIVE_ROOT)

# Set dataset root within Drive (adjust if yours is elsewhere)
DATA_ROOT = f"{DRIVE_ROOT}/MyDrive/mps_detection.v2-micro.coco"
TRAIN_IMAGES = f"{DATA_ROOT}/train"
TRAIN_ANN = f"{TRAIN_IMAGES}/_annotations.coco.json"
VAL_IMAGES = f"{DATA_ROOT}/valid"
VAL_ANN = f"{VAL_IMAGES}/_annotations.coco.json"

# Where to save trained weights in Drive
OUTPUT_DIR = f"{DRIVE_ROOT}/MyDrive/microplastic-ml-colab-outputs"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
# Write the training script to /content/train_faster_rcnn_coco.py
script_path = "/content/train_faster_rcnn_coco.py"
script = r'''import argparse
import json
import os
from pathlib import Path

import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F

try:
    from pycocotools.coco import COCO  # type: ignore
except Exception as e:
    raise SystemExit("pycocotools is required. Install with: pip install pycocotools")


class CocoDetectionDataset(torch.utils.data.Dataset):
    def __init__(self, images_dir: Path, annotation_file: Path, class_names=None):
        self.images_dir = Path(images_dir)
        self.coco = COCO(str(annotation_file))
        self.image_ids = list(self.coco.imgs.keys())
        cats = self.coco.loadCats(self.coco.getCatIds())
        if class_names:
            self.class_names = class_names
            name_to_index = {n: i for i, n in enumerate(class_names)}
            missing = [c['name'] for c in cats if c['name'] not in name_to_index]
            if missing:
                raise RuntimeError(f"COCO categories missing from supplied class list: {missing}")
        else:
            cats_sorted = sorted(cats, key=lambda c: c['id'])
            self.class_names = [c['name'] for c in cats_sorted]
        self.cat_id_to_label = {}
        for c in cats:
            if c['name'] not in self.class_names:
                continue
            label_index = self.class_names.index(c['name']) + 1
            self.cat_id_to_label[c['id']] = label_index

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        from PIL import Image
        image_id = self.image_ids[idx]
        img_info = self.coco.loadImgs([image_id])[0]
        file_name = img_info['file_name']
        path = self.images_dir / file_name
        img = Image.open(path).convert('RGB')
        ann_ids = self.coco.getAnnIds(imgIds=[image_id], iscrowd=None)
        anns = self.coco.loadAnns(ann_ids)
        boxes, labels, areas, iscrowd = [], [], [], []
        for ann in anns:
            if 'bbox' not in ann:
                continue
            x, y, bw, bh = ann['bbox']
            x1, y1, x2, y2 = x, y, x + bw, y + bh
            cat_id = ann['category_id']
            if cat_id not in self.cat_id_to_label:
                continue
            label = self.cat_id_to_label[cat_id]
            boxes.append([x1, y1, x2, y2])
            labels.append(label)
            areas.append(bw * bh)
            iscrowd.append(ann.get('iscrowd', 0))
        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)
        areas = torch.tensor(areas, dtype=torch.float32)
        iscrowd = torch.tensor(iscrowd, dtype=torch.int64)
        if boxes.ndim == 1:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
        if labels.ndim == 0:
            labels = torch.zeros((0,), dtype=torch.int64)
        image_id_tensor = torch.tensor([image_id])
        target = {'boxes': boxes, 'labels': labels, 'image_id': image_id_tensor, 'area': areas, 'iscrowd': iscrowd}
        return torchvision.transforms.functional.to_tensor(img), target


def collate_fn(batch):
    return tuple(zip(*batch))


def create_model(num_classes: int, pretrained_backbone: bool):
    if pretrained_backbone:
        model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(weights="DEFAULT")
    else:
        model = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


def parse_args():
    ap = argparse.ArgumentParser(description='Train Faster R-CNN on COCO-format dataset')
    ap.add_argument('--train-images', required=True)
    ap.add_argument('--train-ann', required=True)
    ap.add_argument('--val-images', required=True)
    ap.add_argument('--val-ann', required=True)
    ap.add_argument('--classes-json', default=None)
    ap.add_argument('--epochs', type=int, default=40)
    ap.add_argument('--batch', type=int, default=2)
    ap.add_argument('--lr', type=float, default=0.005)
    ap.add_argument('--workers', type=int, default=2)
    ap.add_argument('--name', default='rcnn_coco_v1')
    ap.add_argument('--pretrained', action='store_true')
    ap.add_argument('--log-file', default=None)
    ap.add_argument('--resume', action='store_true')
    return ap.parse_args()


def main():
    args = parse_args()
    if args.classes_json and Path(args.classes_json).exists():
        class_names = json.loads(Path(args.classes_json).read_text(encoding='utf-8'))
    else:
        class_names = None
    train_ds = CocoDetectionDataset(Path(args.train_images), Path(args.train_ann), class_names)
    if class_names is None:
        class_names = train_ds.class_names
    val_ds = CocoDetectionDataset(Path(args.val_images), Path(args.val_ann), class_names)
    num_classes = len(class_names) + 1
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=args.batch, shuffle=True, num_workers=args.workers, collate_fn=collate_fn)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=args.batch, shuffle=False, num_workers=args.workers, collate_fn=collate_fn)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = create_model(num_classes, pretrained_backbone=args.pretrained).to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=args.lr, momentum=0.9, weight_decay=0.0005)
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)
    runs_dir = Path('runs') / 'rcnn_coco' / args.name
    runs_dir.mkdir(parents=True, exist_ok=True)
    best_path = runs_dir / 'best.pth'
    last_path = runs_dir / 'last.pth'
    state_file = runs_dir / 'state.pt'
    best_val = float('inf')
    start_epoch = 1
    log_path = Path(args.log_file).resolve() if args.log_file else None
    if log_path:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        with open(log_path, 'a', encoding='utf-8') as lf:
            lf.write(f"# start name={args.name} epochs={args.epochs} classes={class_names}\n")
    if args.resume and state_file.exists() and last_path.exists():
        try:
            ckpt = torch.load(last_path, map_location=device)
            model.load_state_dict(ckpt, strict=False)
            state = torch.load(state_file, map_location=device)
            if 'optimizer' in state:
                optimizer.load_state_dict(state['optimizer'])
            if 'best_val' in state:
                best_val = state['best_val']
            if 'epoch' in state:
                start_epoch = int(state['epoch']) + 1
            line = f"Resumed from epoch {start_epoch-1} (best_val={best_val:.4f})"
            print(line)
            if log_path:
                with open(log_path, 'a', encoding='utf-8') as lf:
                    lf.write(line + '\n')
        except Exception as e:
            print('Resume failed, starting fresh:', e)
    for epoch in range(start_epoch, args.epochs + 1):
        model.train(); total_loss = 0.0
        for images, targets in train_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()
        lr_scheduler.step()
        val_loss = 0.0
        model.train()
        with torch.no_grad():
            for images, targets in val_loader:
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
                loss_dict = model(images, targets)
                loss = sum(loss for loss in loss_dict.values())
                val_loss += loss.item()
        avg_train = total_loss / max(len(train_loader), 1)
        avg_val = val_loss / max(len(val_loader), 1)
        line = f"Epoch {epoch:03d}/{args.epochs} | train_loss: {avg_train:.4f} | val_loss: {avg_val:.4f}"
        print(line, flush=True)
        if log_path:
            with open(log_path, 'a', encoding='utf-8') as lf:
                lf.write(line + '\n')
        torch.save(model.state_dict(), last_path)
        torch.save({'epoch': epoch, 'optimizer': optimizer.state_dict(), 'best_val': best_val}, state_file)
        if avg_val < best_val:
            best_val = avg_val
            torch.save(model.state_dict(), best_path)
    models_dir = Path('microplastic-ml') / 'models'
    models_dir.mkdir(parents=True, exist_ok=True)
    target = models_dir / 'rcnn_coco_best.pth'
    if best_path.exists():
        import shutil
        shutil.copy2(best_path, target)
        print(f'Copied best COCO RCNN weights to {target}')
    else:
        print('WARNING: best.pth not found; no copy performed')
    (models_dir / 'rcnn_coco_classes.json').write_text(json.dumps(class_names, indent=2))

if __name__ == '__main__':
    main()
'''
open(script_path, 'w', encoding='utf-8').write(script)
print("Wrote:", script_path)


In [ ]:
# Run COCO Faster R-CNN training
%cd /content
!python /content/train_faster_rcnn_coco.py \
  --train-images "$TRAIN_IMAGES" \
  --train-ann    "$TRAIN_ANN" \
  --val-images   "$VAL_IMAGES" \
  --val-ann      "$VAL_ANN" \
  --epochs 40 \
  --batch 4 \
  --pretrained \
  --name rcnn_coco_colab \
  --log-file logs/rcnn_coco_colab.txt


In [ ]:
# Copy outputs from /content/microplastic-ml/models to Drive
import shutil, json, pathlib
src_dir = pathlib.Path("/content/microplastic-ml/models")
src_dir.mkdir(parents=True, exist_ok=True)
src_best = src_dir/"rcnn_coco_best.pth"
src_classes = src_dir/"rcnn_coco_classes.json"
dst_best = pathlib.Path(OUTPUT_DIR)/"rcnn_coco_best.pth"
dst_classes = pathlib.Path(OUTPUT_DIR)/"rcnn_coco_classes.json"
if src_best.exists():
    shutil.copy2(src_best, dst_best)
    print("Saved:", dst_best)
else:
    print("best.pth not found at", src_best)
if src_classes.exists():
    shutil.copy2(src_classes, dst_classes)
    print("Saved:", dst_classes)
else:
    print("classes.json not found at", src_classes)
print("Done.")
